# VisionAid — Model Training and Evaluation

**ENSF 444 Final Project**

This notebook trains, tunes, and compares four scikit-learn classifiers on the
VisionAid landmark feature dataset, then connects the best model's predictions
to the eyewear frame recommendation layer.

## How to run

1. Ensure `data/processed/landmark_features.csv` exists.
   If not, run `python run_project.py --force-features` first.
2. Launch Jupyter from the project root and open this file.
3. Run all cells with **Kernel → Restart & Run All**.

**Note:** The GridSearchCV training cells can take 5–15 minutes depending on
your hardware.  Pre-trained models are also available in `models/*.joblib`.

## Contents

1. Setup and data loading
2. Train/test split
3. Model training with hyperparameter tuning
4. Model comparison table
5. Confusion matrices
6. Classification reports
7. Random Forest feature importance
8. Eyewear recommendation examples

## 1. Setup and data loading

In [ ]:
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Make the project src/ directory importable
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from visionaid.data_loader import (
    load_features,
    get_feature_columns,
    LABEL_COLUMN,
    CLASSES,
)
from visionaid.recommendation import get_recommendation

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

# Fixed random seed for reproducibility across all randomized steps
RANDOM_STATE = 42

In [ ]:
# Load the pre-computed landmark + geometry feature CSV
CSV_PATH = PROJECT_ROOT / "data" / "processed" / "landmark_features.csv"
df = load_features(csv_path=CSV_PATH)

# Separate the feature matrix (X) and the target vector (y)
feature_cols = get_feature_columns(df)
X = df[feature_cols]
y = df[LABEL_COLUMN]

print(f"Feature matrix shape: {X.shape}")
print(f"Label distribution:\n{y.value_counts().sort_index()}")

## 2. Train / test split

We hold out **20 %** of the data as a test set.  The split is stratified so
each class is proportionally represented in both splits.

In [ ]:
# Stratified split — ensures each class appears in the same proportion in
# both train and test sets, which matters for this relatively small dataset.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print()
print("Train class distribution:")
print(y_train.value_counts().sort_index())
print()
print("Test class distribution:")
print(y_test.value_counts().sort_index())

## 3. Model training with hyperparameter tuning

We compare four classifiers:

| Model | Type | Notes |
|---|---|---|
| Logistic Regression | Linear | Baseline; requires feature scaling |
| K-Nearest Neighbours | Non-linear | Distance-based; sensitive to scale |
| Random Forest | Non-linear | Ensemble of trees; scale-invariant |
| MLP Classifier | Non-linear | Shallow neural network |

Each model is wrapped in a `sklearn.pipeline.Pipeline` that applies
`SimpleImputer` (NaN safety) and `StandardScaler` where required.

Hyperparameter tuning uses `GridSearchCV` with **5-fold stratified cross-validation**
on the training set.  The best parameter combination is then re-fitted on all
of `X_train` before evaluation on `X_test`.

In [ ]:
# 5-fold stratified cross-validation splitter used inside GridSearchCV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Dictionary mapping model names to their sklearn Pipeline and parameter grid.
# The "model__" prefix is required by Pipeline to route parameters to the
# correct step.
MODEL_CONFIGS = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer()),
            ("scaler", StandardScaler()),     # required for gradient convergence
            ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
        ]),
        "params": {
            "model__C": [0.3, 1.0, 3.0, 10.0],       # inverse regularization strength
            "model__class_weight": [None, "balanced"],  # balanced up-weights minority classes
        },
    },
    "KNN": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer()),
            ("scaler", StandardScaler()),   # essential: KNN uses Euclidean distance
            ("model", KNeighborsClassifier()),
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9, 11],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],   # p=1 → Manhattan, p=2 → Euclidean
        },
    },
    "Random Forest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer()),
            # No scaler needed — decision trees are scale-invariant
            ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
        ]),
        "params": {
            "model__n_estimators": [200, 400],         # number of trees in the ensemble
            "model__max_depth": [None, 12, 20],        # None = grow until pure leaves
            "model__min_samples_leaf": [1, 2, 4],      # controls overfitting
        },
    },
    "MLP Classifier": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer()),
            ("scaler", StandardScaler()),   # required for stable gradient descent
            ("model", MLPClassifier(max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        "params": {
            "model__hidden_layer_sizes": [(64,), (128, 64)],   # network architecture
            "model__alpha": [0.0001, 0.001],                   # L2 regularization
        },
    },
}

In [ ]:
# Train each model and collect evaluation metrics.
# This cell runs GridSearchCV for all four models — expect 5–15 minutes.

results = []
fitted_models = {}  # store fitted estimators for later analysis
test_predictions = {}  # store test-set predictions

for model_name, config in MODEL_CONFIGS.items():
    print(f"Training {model_name}...", end=" ", flush=True)

    # GridSearchCV exhaustively evaluates all parameter combinations
    # using cross-validation on the training set.
    search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring="accuracy",
        n_jobs=1,       # single-threaded for determinism
        refit=True,     # re-trains best params on all of X_train
    )
    search.fit(X_train, y_train)

    # Evaluate the best estimator on the held-out test set
    best_est = search.best_estimator_
    y_pred = best_est.predict(X_test)

    fitted_models[model_name] = best_est
    test_predictions[model_name] = y_pred

    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "Model": model_name,
        "Best CV Accuracy": round(search.best_score_, 4),
        "Test Accuracy": round(test_acc, 4),
        "Weighted F1": round(test_f1, 4),
        "Best Params": search.best_params_,
    })

    print(f"done. Test accuracy: {test_acc:.4f}")

print("\nAll models trained.")

## 4. Model comparison

In [ ]:
# Build a comparison DataFrame sorted by descending test accuracy
results_df = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

# Display without the Best Params column for a cleaner view
display(results_df[["Model", "Best CV Accuracy", "Test Accuracy", "Weighted F1"]])

In [ ]:
# Grouped bar chart — CV accuracy vs test accuracy per model
plot_df = results_df[["Model", "Best CV Accuracy", "Test Accuracy"]].melt(
    id_vars="Model", var_name="Split", value_name="Accuracy"
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=plot_df,
    x="Model",
    y="Accuracy",
    hue="Split",
    palette="Set2",
    ax=ax,
)
ax.set_ylim(0, 1.0)
ax.set_title("Model Comparison: CV Accuracy vs Test Accuracy", fontsize=13)
ax.set_xlabel("")
ax.set_ylabel("Accuracy")
ax.legend(title="")

# Add value labels on top of each bar
for patch in ax.patches:
    height = patch.get_height()
    if height > 0:
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            height + 0.005,
            f"{height:.3f}",
            ha="center", va="bottom", fontsize=9,
        )

plt.tight_layout()
plt.show()

In [ ]:
# Print the best hyperparameters for each model
print("Best hyperparameters per model:")
for row in results:
    print(f"  {row['Model']}: {row['Best Params']}")

## 5. Confusion matrices

A confusion matrix shows where each model makes errors.  Off-diagonal cells
indicate misclassifications.  Rows are true labels; columns are predicted labels.

In [ ]:
labels = sorted(y_test.unique())

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes_flat = axes.flatten()

for i, (model_name, y_pred) in enumerate(test_predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    acc = accuracy_score(y_test, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=axes_flat[i],
    )
    axes_flat[i].set_title(f"{model_name}\n(Test Acc: {acc:.4f})", fontsize=11)
    axes_flat[i].set_xlabel("Predicted")
    axes_flat[i].set_ylabel("True")

fig.suptitle("Confusion Matrices — All Models", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Per-class classification report

The classification report breaks down precision, recall, and F1-score
for each of the five face-shape classes.

In [ ]:
# Show the classification report for the best model
best_model_name = results_df.iloc[0]["Model"]
best_y_pred = test_predictions[best_model_name]

print(f"Best model: {best_model_name}")
print()
print(classification_report(y_test, best_y_pred, zero_division=0))

In [ ]:
# Visualise per-class F1 scores as a horizontal bar chart for all models
fig, axes = plt.subplots(1, len(test_predictions), figsize=(16, 4), sharey=True)

for ax, (model_name, y_pred) in zip(axes, test_predictions.items()):
    # Extract per-class F1 from the classification report dict
    report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    class_f1 = {cls: report[cls]["f1-score"] for cls in labels}

    ax.barh(list(class_f1.keys()), list(class_f1.values()), color="steelblue", alpha=0.8)
    ax.set_xlim(0, 1.0)
    ax.axvline(0.5, color="red", linestyle="--", linewidth=0.8)
    ax.set_title(model_name, fontsize=10)
    ax.set_xlabel("F1-score")

fig.suptitle("Per-Class F1-Score by Model", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Random Forest feature importance

Random Forest computes the mean decrease in impurity (MDI) for each feature
across all trees.  Higher importance means the feature was more useful for
splitting nodes in the forest.

In [ ]:
rf_pipeline = fitted_models["Random Forest"]

# Extract the Random Forest estimator from the pipeline step named "model"
rf_model = rf_pipeline.named_steps["model"]
importances = rf_model.feature_importances_

# Build a ranked DataFrame of the top 15 features
importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

print("Top 15 Random Forest features:")
display(importance_df)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.barplot(
    data=importance_df,
    x="importance",
    y="feature",
    hue="feature",
    palette="mako",
    legend=False,
    ax=ax,
)
ax.set_title("Top 15 Random Forest Feature Importances (MDI)", fontsize=13)
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 8. Eyewear recommendation examples

The final stage maps each predicted face shape to a shortlist of eyewear frame
styles using the hand-designed recommendation rules in `src/visionaid/recommendation.py`.

In [ ]:
# Generate recommendations for the first 15 test-set predictions from the best model
recommendation_rows = []

# Align test indices with the original DataFrame for image path lookup
test_index = X_test.index.tolist()

for i, idx in enumerate(test_index[:15]):
    actual = y_test.iloc[i]
    predicted = best_y_pred[i]

    # Look up the three recommended frame styles for the predicted face shape
    rec = get_recommendation(predicted)

    recommendation_rows.append({
        "image_path": df.loc[idx, "image_path"],
        "actual_shape": actual,
        "predicted_shape": predicted,
        "correct": "✓" if actual == predicted else "✗",
        "frame_1": rec.recommended_frames[0],
        "frame_2": rec.recommended_frames[1],
        "frame_3": rec.recommended_frames[2],
    })

rec_df = pd.DataFrame(recommendation_rows)
display(rec_df)

In [ ]:
# Show the full rationale for each unique predicted face shape
print("Recommendation rationale by face shape:\n")
for shape in sorted(rec_df["predicted_shape"].unique()):
    rec = get_recommendation(shape)
    print(f"  {shape.upper()}:")
    print(f"    Frames: {', '.join(rec.recommended_frames)}")
    print(f"    Rationale: {rec.rationale}")
    print()

## Summary

| Model | CV Accuracy | Test Accuracy | Weighted F1 |
|---|---|---|---|
| Logistic Regression | ~0.599 | **~0.626** | **~0.624** |
| Random Forest | ~0.551 | ~0.606 | ~0.603 |
| MLP Classifier | ~0.551 | ~0.596 | ~0.595 |
| KNN | ~0.470 | ~0.475 | ~0.474 |

### Key findings

- **Logistic Regression** is the best-performing model despite being the simplest,
  suggesting the normalized landmark features already form a fairly linearly
  separable feature space.
- **KNN** performs worst — likely because the high-dimensional feature space
  (152 landmark + geometry columns) makes distance metrics less meaningful.
- **`oval`** is consistently the hardest class to classify across all models,
  which makes sense because oval faces share characteristics with all other categories.
- The best test accuracy of ~62.6 % is below commercial product thresholds but
  competitive for a classical ML approach on a 500-image dataset.
- The most important features are geometric (jaw angle, cheekbone ratios),
  which aligns with how opticians and face-shape guides reason about facial structure.